# Uitwerkingen bij het demo-college — Exploratory Data Analysis

Bij `03_demo_college.ipynb`. Per cel: wat de code doet, waar studenten over struikelen,
en waar het aansluit op de pokerlijn.

**Zo gebruik je dit:** draai de cel in het demo-notebook, stel de vraag, en gebruik de tekst
hieronder als je iets wil toevoegen. De cellen hier zijn dezelfde, met de uitkomst erbij.

## Cel 0–1 · Filteren

```python
europa = gapminder[gapminder['continent'] == 'Europe']
```

**Wat er gebeurt.** `gapminder['continent'] == 'Europe'` is geen filter maar een kolom van
`True`/`False` — net zo lang als de tabel. Die zet je tussen de haken, en pandas houdt de
rijen over waar `True` staat.

**De vraag om te stellen.** *Wat krijg je als je alleen `gapminder['continent'] == 'Europe'`
uitvoert?* Laat het zien — het is een Series van 1704 booleans. Dat het filteren twee stappen
is (eerst een masker, dan selecteren) is precies wat studenten mist.

**Waar het terugkomt.** In de pokerlijn doen ze dit constant:
`hand_log[hand_log["aan_zet"]]` — daar is het masker al een boolean-kolom, dus dan zie je de
vorm in zijn kaalste vorm.

In [ ]:
from gapminder import gapminder

masker = gapminder['continent'] == 'Europe'
print(type(masker).__name__, "van", len(masker), "waarden")
print(masker.head(3).to_string())
print("\naantal True:", masker.sum())

## Cel 2 · groupby — en meteen een valkuil

```python
gapminder.groupby('continent')['pop'].sum()
```

**Wat er gebeurt.** Rijen met dezelfde continent-waarde gaan op één hoop, en van elke hoop
wordt de `pop`-kolom opgeteld.

**⚠️ De valkuil, en hij is de moeite waard.** Dit telt op over **alle twaalf jaren**. Europa
komt uit op 6,18 miljard — meer dan de wereldbevolking. Het getal is niet fout gerekend, het
is een verkeerde vraag.

| | |
|---|---|
| Europa, som over alle jaren | 6.181.115.304 |
| Europa in 2007 alleen | 586.098.529 |

**De vraag om te stellen.** *Europa heeft volgens deze uitkomst 6 miljard inwoners. Klopt dat?*
Laat ze zelf vinden waar het misgaat. Dit is het beste moment van de demo: een `groupby` die
netjes draait en een antwoord geeft dat nergens op slaat.

**Waar het terugkomt.** Exact hetzelfde gebeurt bij de pokerbot als je de winst optelt over
alle simulaties in plaats van per simulatie te kijken.

In [ ]:
alle_jaren = gapminder.groupby('continent')['pop'].sum()['Europe']
alleen_2007 = gapminder[(gapminder.continent == 'Europe') & (gapminder.year == 2007)]['pop'].sum()
print(f"over alle jaren : {alle_jaren:>15,}")
print(f"in 2007 alleen  : {alleen_2007:>15,}")
print(f"factor          : {alle_jaren / alleen_2007:>15.1f}x te hoog")

## Cel 3 · Twee niveaus, en waarom `reset_index()`

```python
gapminder.groupby(['continent','year'])['pop'].sum().reset_index()
```

**Wat er gebeurt.** Groeperen op twee kolommen geeft een Series met een MultiIndex —
`continent` en `year` zitten in de *index*, niet in kolommen. `reset_index()` duwt ze terug
naar kolommen, en dan heb je weer een gewone tabel.

**De vraag om te stellen.** *Waarom staat `reset_index()` erachter? Laat hem eens weg.* Zonder
is het een Series met een index van tupels; daar kun je niet mee plotten en niet op mergen.

**Waar het terugkomt.** In Werkcollege 5 Deel 6 mergen ze twee tabellen op een sleutel. Zit die
sleutel in de index in plaats van in een kolom, dan werkt `merge` niet — en dat is de meest
voorkomende vastloper daar.

In [ ]:
zonder = gapminder.groupby(['continent', 'year'])['pop'].sum()
met = zonder.reset_index()
print("zonder reset_index:", type(zonder).__name__, "· index:", zonder.index.names)
print("met reset_index   :", type(met).__name__, "· kolommen:", list(met.columns))

## Cel 4–5 · De widget

```python
widgets.interact(barchart, x = x_dict)
```

**Wat er gebeurt.** `interact` kijkt naar de parameters van je functie en bouwt er een
besturingselement bij. Omdat `x` een dictionary meekrijgt, wordt het een dropdown; bij een
getal zou het een slider zijn geworden.

**Wat ze moeten meenemen.** Niet de syntax van `ipywidgets` — die gebruiken ze verder nergens.
Wel het idee: **een control die een grafiek stuurt.** Dat is precies wat ze in Werkcollege 5
Deel 7 in Streamlit gaan doen, met `st.selectbox` in plaats van `widgets.interact`.

**Let op de selectie van numerieke kolommen.** `select_dtypes(include=numerics)` pakt ook
`year` mee. Een histogram van jaartallen is technisch correct en inhoudelijk onzin — mooie
kleine check of ze meekijken.

## Cel 6 · `apply()` met een lambda

```python
gapminder.groupby('continent').apply(lambda g: "Dit continent heeft " + str(g['country'].nunique()) + " landen")
```

**Wat er gebeurt.** `apply` geeft per groep de hele deeltabel aan je functie. `g` is dus een
compleet DataFrame van één continent, en `nunique()` telt de unieke landen.

**⚠️ In recente pandas krijg je hier een FutureWarning**: *"DataFrameGroupBy.apply operated on
the grouping columns"*. Dat is geen fout in de code van het college maar een aangekondigde
wijziging. Wil je hem weg: `include_groups=False`, of selecteer de kolom vóór de apply.

**De vraag om te stellen.** *Waarom `nunique()` en niet `count()`?* Elk land staat er twaalf
keer in, één keer per jaar. `count()` zou 624 voor Afrika geven in plaats van 52.

**Wanneer gebruik je apply, en wanneer niet.** Hier had het ook gekund met
`groupby('continent')['country'].nunique()` — korter en sneller. `apply` is voor wanneer je
écht de hele groep nodig hebt.

In [ ]:
kort = gapminder.groupby('continent')['country'].nunique()
print("zonder apply, zelfde antwoord:")
print(kort.to_string())
print("\nmet count() in plaats van nunique() — elk land staat er 12x in:")
print(gapminder.groupby('continent')['country'].count().to_string())

## Cel 7 · "Wat doet dit stuk code?"

```python
gapminder.loc[:,~gapminder.columns.str.islower()]
```

**Het antwoord.** Hij houdt de kolommen over waarvan de **naam niet volledig uit kleine letters
bestaat**: `lifeExp` en `gdpPercap`.

Uit elkaar gehaald:

| stuk | wat het is |
|---|---|
| `gapminder.columns` | de kolomnamen, als een Index van strings |
| `.str.islower()` | per naam: staat er alleen kleine letters in? |
| `~` | omkeren — dus: naam met een hoofdletter erin |
| `.loc[:, ...]` | alle rijen, en alleen die kolommen |

**De val.** Het lijkt of hij de numerieke kolommen selecteert, en dat is toevallig bijna waar.
Maar `year` en `pop` zijn óók numeriek en vallen af. Hij selecteert op **schrijfwijze**, niet
op type. Wil je numerieke kolommen, dan gebruik je `select_dtypes` — zoals cel 5 doet.

**De vraag om te stellen.** *Werkt dit nog als iemand de kolom `pop` hernoemt naar `Population`?*
Nee — dan verandert de uitkomst zonder dat de data veranderd is. Code die op een naamconventie
leunt, breekt zodra iemand de conventie niet kent.

In [ ]:
import pandas as pd
overzicht = pd.DataFrame({
    "kolom": gapminder.columns,
    "alleen kleine letters": gapminder.columns.str.islower(),
    "type": [str(gapminder[k].dtype) for k in gapminder.columns],
})
print(overzicht.to_string(index=False))
print("\n~islower() houdt over:", list(gapminder.loc[:, ~gapminder.columns.str.islower()].columns))

## De brug naar de pokerlijn

Drie dingen die je hier kunt leggen, elk in één zin:

**Dezelfde regel, andere data.** `gapminder.groupby('continent')['pop'].sum()` en
`hand_log.groupby('bot_naam')['winst'].sum()` zijn letterlijk dezelfde vorm. Laat ze het
tweede zelf typen nadat ze het eerste hebben gezien.

**Een verdeling máken tegenover een verdeling beschrijven.** De dia's over `np.random.normal`
laten zien hoe je er een genereert; `schat_winkans` gebruikt dat om een kans te schatten die
niet uit te rekenen is. Zelfde gereedschap, andere vraag.

**Het gemiddelde tegenover de mediaan.** Op het toernooi-log is het gemiddelde van de winst per
hand exact nul (nulsom) en de mediaan −20 (de big blind). Eén `describe()` en de hele dia over
centrale tendensen staat op het bord, met hun eigen data.